In [1]:
import numpy as np
from scipy.stats import norm
from scipy.special import gamma
from scipy.optimize import root_scalar

In [2]:
def c_alpha(alpha):
    """Calculates the constant c_alpha."""
    # Handle alpha = 0 as a special case to avoid division by zero
    if alpha == 0:
        return 1.0
    return np.exp(alpha) * (alpha ** -alpha) * gamma(alpha + 1)

def mills_ratio(x):
    """Calculates the Mills ratio M(x) = phi(x) / G(x)."""
    # norm.pdf is phi, norm.sf (survival function) is G
    return norm.pdf(x) / norm.sf(x)

def inverse_mills_ratio(y):
    """Numerically finds M^{-1}(y)."""
    # We find the root of M(x) - y = 0
    # A bracket of [-10, 20] is safely wide enough for standard normal quantiles
    res = root_scalar(lambda x: mills_ratio(x) - y, bracket=[-10, 20])
    return res.root

def tau(alpha):
    """Calculates tau(alpha)."""
    c_val = c_alpha(alpha)
    m_inv_val = inverse_mills_ratio(c_val / np.sqrt(2 * np.pi))
    return np.exp(-0.5 * m_inv_val**2)

def find_alpha_for_delta(target_delta):
    """Finds the optimal alpha for a specific delta threshold."""
    # We find the root of tau(alpha) - target_delta = 0
    # We search in a reasonable bracket for alpha, e.g., [0.001, 50]
    res = root_scalar(lambda a: tau(a) - target_delta, bracket=[1e-3, 50])
    return res.root



In [3]:
# 1. Verify the paper's claim for delta = 0.01 
alpha_01 = find_alpha_for_delta(0.01)
print(f"Alpha for delta = 0.01: {alpha_01:.3f} (Expected: ~10.8)")

# 2. Find the alpha boundary for your specific interest: delta = 0.1
alpha_10 = find_alpha_for_delta(0.1)
print(f"Alpha for delta = 0.10: {alpha_10:.3f}")

Alpha for delta = 0.01: 10.825 (Expected: ~10.8)
Alpha for delta = 0.10: 6.097
